# Weather / Polymarket accuracy bot — запуск проекта

Модульная структура (`src/`):
- `config.py` — загрузка `config/config.yaml` (все параметры, разбитые по секциям: `telegram`, `paths`, `chrome`, `weather_miner`, `actuals_miner`, `accuracy_report`, `scheduler`, `criteria`).
- `cities.py` — единый словарь городов `CITIES`.
- `base_miner.py` — общая инфраструктура Selenium-майнеров (логирование, подключение к Chrome, дедуп ошибок в Telegram).
- `telegram_client.py` — отправка сообщений в Telegram.
- `weather_miner.py` — `WeatherMiner`: майнер исторических данных (WU/Windy/Polymarket через Gamma API), `run_cycle()` — один проход. Каждый источник пишется в свою папку (`wunderground_forcast/`, `windy_forcast/`, `polymarket_gamma/`).
- `polymarket_clob_miner.py` — `PolymarketClobMiner`: реальные bid/ask по CLOB API (без Selenium), 15-минутная сетка 06:00–18:00 по местному времени — пишет в `polymarket_clob/`.
- `actuals_miner.py` — `ActualsMiner`: майнер факта, инкрементальный (не майнит уже смайненное), параметризованный день `target_date`.
- `city_metrics.py` — общие функции метрик (попадания/промахи, череда, цена ставки) — используются и отчётом, и отбором по критериям.
- `accuracy_report.py` — `AccuracyReportBuilder`: построение DataFrame + самодостаточный HTML-отчёт (перезаписывается каждый день).
- `criteria.py` — `CityCriteriaSelector`: отбор городов по трём критериям (попадания/цена ставки/серия промахов).
- `notifier.py` — `QualifyingCitiesNotifier`: сообщение в Telegram + сохранение JSONL с датой.
- `orchestrator.py` — `Orchestrator`: связывает всё вместе — непрерывный майнинг погоды с ежедневной паузой сразу после полуночи по Сиэтлу для факта/отчёта/отбора городов.

**Перед запуском:** подними Chrome с remote debugging (тем же профилем, что и раньше):
```
start chrome.exe --remote-debugging-port=9222 --user-data-dir="C:\bot_chrome_profile"
```
Все параметры (токен Telegram, пороги критериев, расписание, слоты майнинга и т.д.) — в `config/config.yaml`.

## Авто-торговля — переменные окружения

Нужно выполнить ОДИН раз за сессию, ДО `Orchestrator(config)` ниже, только если хотя бы у одного набора (`price_monitor.weather_forecast`/`price_monitor.max_bet`) включено `auto_trading: true` в `config/config.yaml`. Если оба выключены — можно пропустить, бот отработает как обычно (майнинг/отчёты/сигналы в Telegram), просто без реальных сделок. Приватный ключ НИКОГДА не хранится в config.yaml — см. README, раздел "Авто-торговля".

In [ ]:
import os, getpass

os.environ["POLYMARKET_WALLET_ADDRESS"] = getpass.getpass("POLYMARKET_WALLET_ADDRESS")
os.environ["POLYMARKET_PRIVATE_KEY"] = getpass.getpass("POLYMARKET_PRIVATE_KEY")

In [ ]:
import sys
sys.path.insert(0, ".")  # запускать ноутбук из корня проекта (там же папка config/)

from src.config import Config
from src.orchestrator import Orchestrator

config = Config("config/config.yaml")
orchestrator = Orchestrator(config)

## Основной запуск

Бесконечный цикл: майнер погоды идёт непрерывно (`weather_miner.run_cycle()` каждые `scheduler.cycle_sleep_sec` секунд), а раз в сутки в `scheduler.actuals_trigger_hour` по Сиэтлу автоматически ставится на паузу и вместо очередного прохода запускается дневной пайплайн (факт → отчёт → отбор городов → Telegram), после чего майнинг погоды продолжается сам собой. Остановка — `KeyboardInterrupt` (Kernel → Interrupt).

In [ ]:
orchestrator.run()

## Отдельные запуски компонентов (для отладки)

Ниже — то же самое, но по частям, если нужно прогнать что-то одно вручную, не трогая основной цикл выше.

### Один проход майнера погоды

In [ ]:
orchestrator.weather_miner.run_cycle()

### Один проход майнера реальных bid/ask (CLOB API)

In [ ]:
orchestrator.polymarket_clob_miner.run_cycle()

### Майнер факта за конкретный день (по умолчанию — сегодня по Сиэтлу)

In [ ]:
target_date = orchestrator._target_date()
print("target_date:", target_date)
orchestrator.actuals_miner.run(target_date=target_date)

### Факт Polymarket (собственное разрешение события) за месяц — можно перезагрузить любой месяц

In [ ]:
# target_date -- день, до которого домайнить месяц (месяц берётся из этой даты).
# Можно передать и другой месяц, чтобы перезагрузить его целиком:
# orchestrator.polymarket_actuals_miner.run(target_date="2026-08-31")
orchestrator.polymarket_actuals_miner.run(target_date=target_date)

### Построение отчёта и отбор городов по критериям (без Telegram/сохранения — только просмотр)

In [ ]:
df, skip_info = orchestrator.report_builder.build_dataframe()
orchestrator.report_builder.print_skip_details(skip_info)
df.head()

In [ ]:
# Второй вариант — с галочкой "1 час с максимумом не учитывается" в отчётах
df_effective, _ = orchestrator.report_builder.build_dataframe(effective_max=True)
dfs = {"raw": df, "effective": df_effective}

out_path = orchestrator.report_builder.save_report(dfs)
print("Отчёт сохранён:", out_path)

qualifying = orchestrator.criteria_selector.select(df)  # отбор по критериям — всегда по сырому максимуму
qualifying

### Второй отчёт — только watched_icaos_weather_forecast.yaml (поправка/цена уже применены из конфига)

In [ ]:
watched_out_path = orchestrator.report_builder.save_watched_report(dfs, orchestrator.watched)
print("Скорректированный отчёт сохранён:", watched_out_path) if watched_out_path \
    else print("Отчёт не построен — нет данных по городам из watched_icaos.yaml")

### Третий отчёт — только watched_icaos_max_bet.yaml (без прогноза погоды, таргет = самая дорогая ставка)

In [ ]:
max_bet_out_path = orchestrator.report_builder.save_market_target_report(
    orchestrator.watched_max_bet, orchestrator.config.price_monitor.monitor_slot,
)
print("Отчёт по самой дорогой ставке сохранён:", max_bet_out_path) if max_bet_out_path \
    else print("Отчёт не построен — нет данных по городам из watched_icaos_max_bet.yaml")

### Ручная отправка отобранных городов в Telegram + сохранение JSONL

In [ ]:
saved_path = orchestrator.notifier.send_and_save(qualifying, run_date=target_date)
print("Сохранено:", saved_path)

### Полный дневной пайплайн вручную (факт + отчёт + отбор + Telegram), без ожидания полуночи по Сиэтлу

In [ ]:
orchestrator._run_daily_pipeline()

### Авто-торговля: текущее состояние (открытые позиции / промахи пирамиды / risk-halt)

In [ ]:
import json

store = orchestrator.trading_engine.store
print("Открытых позиций:", len(store.open_positions))
for pos in store.open_positions.values():
    print(f"  {pos['icao']} {pos['strategy_set']} лот={pos['lot']} статус={pos['order_status']} открыта={pos['opened_at']}")
print()
print("Промахи подряд (пирамида):", store.pyramid_state)
print("Risk-состояние:", json.dumps(store.risk_state, ensure_ascii=False, indent=2))

### Авто-торговля: разрешить открытые позиции по факту вручную (без ожидания дневного пайплайна)

In [ ]:
orchestrator.trading_engine.resolve_positions()

### Авто-торговля: возобновить остановленную риск-менеджментом стратегию (ПОСЛЕ ручной проверки причины!)

`halt_key` — `"weather_forecast"` / `"max_bet"` (одна стратегия) или `"_total"` (вся торговля).

In [ ]:
halt_key = "max_bet"  # поменяй на нужный ключ перед запуском

store = orchestrator.trading_engine.store
if halt_key in store.risk_state:
    store.risk_state[halt_key]["halted"] = False
    store.save_risk_state()
    print(f"{halt_key}: халт снят.")
else:
    print(f"{halt_key}: халта и не было.")